# Week 04B — Regions and Shape Analysis

**MCTA 4364 Machine Vision** · Session 2 of Week 4 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W04_segmentation/W04B_regions_shape_analysis.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W04_segmentation/W04B_regions_shape_analysis.ipynb)

A binary mask is only a set of pixels. To **count** parts, **measure** them, **classify** them and **find defects** we must turn the mask into a list of
**objects** and describe each one with numbers. This session takes you from pixels to objects to measurements, the core of most classical
inspection systems.

### Learning outcomes
By the end of this session you will be able to:
1. Label **connected components** (4- vs 8-connectivity) and read their statistics.
2. Segment by **region growing** and by **clustering** (k-means in colour space).
3. Separate **touching objects** with the distance transform and the **watershed** algorithm.
4. Extract **contours** and use the contour **hierarchy** (e.g. to count holes).
5. Compute **shape descriptors** (area, perimeter, circularity, solidity, moments, Hu invariants) and use them to classify parts and find **notches**.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–15 | 1–2 | Connected components, region growing, k-means |
| 15–32 | 3 | Touching objects: distance transform and watershed (3D view) |
| 32–45 | 4 | Contours and hierarchy |
| 45–68 | 5–6 | Shape descriptors, classification, defect detection, coin case study |
| 68–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import skimage.data

from cvhelpers import check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
rng = np.random.default_rng(0)
coins = skimage.data.coins()
fruits = load_sample("fruits.jpg")


def colour_labels(labels):
    """Random colour per label (0 = black background) for display."""
    palette = np.vstack([[0, 0, 0], np.random.default_rng(1).integers(60, 255, (max(labels.max(), 1), 3))]).astype(np.uint8)
    return palette[np.clip(labels, 0, None)]

## 1. Connected components

Two foreground pixels belong to the same object if a path of foreground pixels joins them. Whether *diagonal* neighbours count is the
**connectivity**: **4-connectivity** (up, down, left, right) or **8-connectivity** (also diagonals). The choice changes the answer on thin,
diagonal structures. `cv2.connectedComponentsWithStats` labels every object and returns its **area**, **bounding box** and **centroid** in one pass.

In [ ]:
grid = np.zeros((9, 12), np.uint8)
grid[1:4, 1:4] = 1; grid[4, 4] = 1; grid[5, 5] = 1; grid[6, 6:10] = 1     # a blob connected to a line only diagonally
grid[1:3, 8:11] = 1
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
axes[0].imshow(grid, cmap="Greys", vmin=0, vmax=1.4); axes[0].set_title("binary input")
for ax, conn in zip(axes[1:], (4, 8)):
    n, lab = cv2.connectedComponents(grid, connectivity=conn)
    ax.imshow(colour_labels(lab)); ax.set_title(f"{conn}-connectivity: {n - 1} objects")
for ax in axes:
    ax.set_xticks(np.arange(-0.5, 12, 1), minor=True); ax.set_yticks(np.arange(-0.5, 9, 1), minor=True)
    ax.grid(which="minor", color="#dadce0"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Figure 1.1 - Connectivity decides whether diagonal pixels join objects", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
_, coins_bin = cv2.threshold(cv2.GaussianBlur(coins, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
coins_bin = cv2.morphologyEx(coins_bin, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
n, labels, stats, cents = cv2.connectedComponentsWithStats(coins_bin, connectivity=8)
vis = cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR)
for i in range(1, n):
    x, y, w, h, area = stats[i]
    if area < 150:
        continue
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 200, 255), 1)
    cv2.putText(vis, str(area), (x, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 0), 1)
show(coins_bin, colour_labels(labels), vis, titles=["Otsu + opening", f"{n - 1} components (random colours)",
                                                   "bounding boxes and areas (> 150 px)"], figsize=(21, 5))
print("Some coins merge with the bright top border, and some have dark gaps: the next sections deal with both problems.")

## 2. Region growing and clustering

**Region growing (flood fill)** starts from a **seed** pixel and repeatedly adds neighbours whose value is within a tolerance of the seed (or of the
growing region). It segments exactly one object, chosen by the user or by another algorithm: ideal for interactive tools and for "select the part
under the robot's gripper".

**Clustering** ignores pixel positions and groups pixels by **feature similarity**, e.g. colour. **k-means** alternates two steps until stable:
assign each pixel to the nearest of $k$ cluster centres, then move each centre to the mean of its pixels. It minimises
$\sum_i \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c(i)} \rVert^2$. You must choose $k$, and clusters are not guaranteed to be spatially connected.

In [ ]:
def flood_demo(seed_x=300, seed_y=250, tolerance=20):
    img = fruits.copy()
    mask = np.zeros((img.shape[0] + 2, img.shape[1] + 2), np.uint8)
    cv2.floodFill(img, mask, (seed_x, seed_y), (255, 0, 255), (tolerance,) * 3, (tolerance,) * 3, cv2.FLOODFILL_FIXED_RANGE)
    cv2.circle(img, (seed_x, seed_y), 8, (0, 255, 255), -1)
    area = int(mask[1:-1, 1:-1].sum())
    show(img, titles=[f"flood fill from the yellow seed, tolerance +/-{tolerance}: {area} px"], figsize=(8, 6))


interact(flood_demo, seed_x=widgets.IntSlider(value=300, min=0, max=fruits.shape[1] - 1), seed_y=widgets.IntSlider(value=250, min=0, max=fruits.shape[0] - 1),
         tolerance=widgets.IntSlider(value=20, min=1, max=80))

In [ ]:
lab_img = cv2.cvtColor(cv2.GaussianBlur(fruits, (5, 5), 0), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
axes[0].imshow(cv2.cvtColor(fruits, cv2.COLOR_BGR2RGB)); axes[0].set_title("input")
for ax, k in zip(axes[1:], (3, 5, 8)):
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    compactness, lab_ids, centres = cv2.kmeans(lab_img, k, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    centres_bgr = cv2.cvtColor(centres.astype(np.uint8)[None], cv2.COLOR_LAB2BGR)[0]
    seg = centres_bgr[lab_ids.ravel()].reshape(fruits.shape)
    ax.imshow(cv2.cvtColor(seg, cv2.COLOR_BGR2RGB)); ax.set_title(f"k-means in Lab, k = {k} (compactness {compactness:.2e})")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 2.1 - Colour clustering: every pixel is replaced by its cluster centre", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Separating touching objects: distance transform and watershed

Touching objects merge into one component. Two classic tools split them:

1. The **distance transform** replaces every foreground pixel with its distance to the nearest background pixel. Each round object becomes a
   **hill** whose peak is at its centre, even when objects touch.
2. The **watershed** algorithm treats an image as a landscape and "floods" it from **markers** (one per object). Where water from two markers meets,
   it builds a **dam**: the boundary between objects. Flooding the *negative* distance transform from markers placed at the hill tops splits touching objects
   along their "necks".

Without good markers, watershed **over-segments** (every noise minimum becomes a region), so marker selection is the key step.

In [ ]:
blobs = np.zeros((260, 360), np.uint8)
for c, r in [((90, 130), 55), ((180, 110), 50), ((262, 150), 45), ((310, 65), 32), ((170, 205), 38)]:
    cv2.circle(blobs, c, r, 255, -1)
dist = cv2.distanceTransform(blobs, cv2.DIST_L2, 5)
_, peaks = cv2.threshold(dist, 0.55 * dist.max(), 255, cv2.THRESH_BINARY)
n_mark, markers = cv2.connectedComponents(peaks.astype(np.uint8))
sure_bg = cv2.dilate(blobs, np.ones((3, 3), np.uint8), iterations=3)
markers = markers + 1
markers[(sure_bg > 0) & (peaks == 0)] = 0                                  # unknown region: to be decided by flooding
ws = cv2.watershed(cv2.cvtColor(blobs, cv2.COLOR_GRAY2BGR), markers.copy())
n_naive = cv2.connectedComponents(blobs)[0] - 1
noisy_dist = dist + rng.normal(0, 0.4, dist.shape).astype(np.float32)        # a little noise, as in any real image
local_max = ((cv2.dilate(noisy_dist, np.ones((5, 5))) == noisy_dist) & (blobs > 0)).astype(np.uint8)
mk_over = cv2.connectedComponents(local_max)[1] + 1
mk_over[(sure_bg > 0) & (local_max == 0)] = 0
over = cv2.watershed(cv2.cvtColor(blobs, cv2.COLOR_GRAY2BGR), mk_over.astype(np.int32))
fig, axes = plt.subplots(1, 5, figsize=(25, 4.4))
axes[0].imshow(blobs, cmap="gray"); axes[0].set_title(f"5 touching discs: connected components = {n_naive}")
axes[1].imshow(dist, cmap="magma"); axes[1].set_title("distance transform: one hill per disc")
axes[2].imshow(peaks, cmap="gray"); axes[2].set_title(f"markers: hill tops (> 55% of max): {n_mark - 1}")
axes[3].imshow(colour_labels(np.where(ws > 1, ws - 1, 0))); axes[3].set_title(f"watershed with markers: {len(np.unique(ws)) - 2} objects")
axes[4].imshow(colour_labels(np.where(over > 1, over - 1, 0))); axes[4].set_title(f"every local maximum as a marker: {len(np.unique(over)) - 2} (over-segmented)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 3.1 - Marker-controlled watershed splits touching objects", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
d = cv2.resize(dist, (180, 130))
fig = go.Figure(go.Surface(z=d, colorscale="Magma", showscale=False))
fig.update_layout(title="Figure 3.2 - The distance transform as a landscape (drag to rotate): watershed floods the valleys between the hills",
                  height=500, scene=dict(zaxis_title="distance to background", aspectratio=dict(x=1.4, y=1, z=0.5)))
fig.show()

### Interactive: marker threshold
A low threshold merges objects (one marker for two discs); a high one can lose small discs. Find the range that gives exactly 5.

In [ ]:
def watershed_demo(marker_fraction=0.55):
    _, pk = cv2.threshold(dist, marker_fraction * dist.max(), 255, cv2.THRESH_BINARY)
    nm, mk = cv2.connectedComponents(pk.astype(np.uint8))
    mk = mk + 1
    mk[(sure_bg > 0) & (pk == 0)] = 0
    res = cv2.watershed(cv2.cvtColor(blobs, cv2.COLOR_GRAY2BGR), mk)
    show(pk.astype(np.uint8), colour_labels(np.where(res > 1, res - 1, 0)),
         titles=[f"markers: {nm - 1}", f"watershed: {len(np.unique(res)) - 2} objects"], figsize=(12, 4.5))


interact(watershed_demo, marker_fraction=widgets.FloatSlider(value=0.55, min=0.1, max=0.95, step=0.05))

## 4. Contours and their hierarchy

A **contour** is the ordered list of boundary points of a region: a compact representation used for measuring, drawing and matching shapes.
`cv2.findContours` also returns a **hierarchy**: for every contour, `[next, previous, first_child, parent]`. With `RETR_TREE` an outer boundary's
**children are its holes**, and objects inside holes are grandchildren. This lets us count holes in a part without any extra processing.

`CHAIN_APPROX_SIMPLE` stores only the corner points of straight runs, and `approxPolyDP` simplifies a contour to a polygon within a tolerance ε
(Douglas–Peucker), which is useful for counting vertices.

In [ ]:
parts = np.zeros((260, 520), np.uint8)
cv2.rectangle(parts, (20, 40), (160, 220), 255, -1); cv2.circle(parts, (60, 90), 18, 0, -1); cv2.circle(parts, (120, 170), 25, 0, -1)
cv2.circle(parts, (120, 170), 10, 255, -1)                                # a pin inside a hole
cv2.circle(parts, (270, 130), 80, 255, -1); cv2.circle(parts, (270, 130), 30, 0, -1)   # a washer
cv2.fillPoly(parts, [np.array([[400, 40], [500, 120], [430, 220], [370, 130]])], 255)  # a plate without holes
contours, hierarchy = cv2.findContours(parts, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
hierarchy = hierarchy[0]
depth = []
for i in range(len(contours)):
    d_, p_ = 0, hierarchy[i][3]
    while p_ != -1:
        d_ += 1; p_ = hierarchy[p_][3]
    depth.append(d_)
vis = cv2.cvtColor(parts, cv2.COLOR_GRAY2BGR) // 3
level_colours = [(0, 200, 0), (0, 0, 255), (255, 160, 0)]
for i, c in enumerate(contours):
    cv2.drawContours(vis, contours, i, level_colours[depth[i] % 3], 2)
    x, y = c[0][0]
    cv2.putText(vis, str(i), (int(x) + 3, int(y) - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
fig, axes = plt.subplots(1, 2, figsize=(17, 4.6), gridspec_kw={"width_ratios": [2, 1]})
axes[0].imshow(vis[..., ::-1]); axes[0].axis("off")
axes[0].set_title("contours coloured by depth: green = outer, red = hole, orange = object inside a hole")
axes[1].axis("off")
rows = [["#", "parent", "first child", "depth", "role"]]
for i, (nx, pv, ch, pa) in enumerate(hierarchy):
    rows.append([i, pa, ch, depth[i], ["outer boundary", "hole", "part inside hole"][depth[i] % 3]])
tbl = axes[1].table(cellText=rows[1:], colLabels=rows[0], loc="center"); tbl.scale(1, 1.4)
axes[1].set_title("hierarchy table")
plt.suptitle("Figure 4.1 - The contour tree encodes containment: holes are children of outer boundaries", fontweight="bold")
plt.tight_layout(); plt.show()
for i in range(len(contours)):
    if depth[i] == 0:
        holes = sum(1 for j in range(len(contours)) if hierarchy[j][3] == i)
        print(f"part {i}: {holes} hole(s)")

## 5. Shape descriptors

| Descriptor | Formula / OpenCV | Invariant to | Use |
|---|---|---|---|
| area $A$ | `contourArea` | rotation, translation | size, sorting |
| perimeter $P$ | `arcLength` | rotation, translation | roughness |
| **circularity** | $4\pi A / P^2$ (1 for a circle) | + scale | round vs angular parts |
| aspect ratio | $w/h$ of `minAreaRect` | + scale | elongation (pins, bolts) |
| extent | $A / A_\text{bbox}$ | + scale | how box-like |
| **solidity** | $A / A_\text{convex hull}$ | + scale | concavities, notches, cracks |
| equivalent diameter | $\sqrt{4A/\pi}$ | rotation | size in mm after calibration |
| orientation | `fitEllipse`, `minAreaRect` or moments | | robot grasp angle |
| **Hu moments** $\phi_1 \dots \phi_7$ | `HuMoments(moments(c))` | rotation, scale, translation | shape matching (`matchShapes`) |

**Image moments** summarise a shape: $m_{pq} = \sum_{x,y} x^p y^q I(x, y)$. The centroid is $(m_{10}/m_{00},\, m_{01}/m_{00})$; central moments $\mu_{pq}$ are
taken about it, normalised moments add scale invariance, and Hu (1962) combined them into seven rotation-invariant values.

In [ ]:
def make_shape(kind, size, angle, center=(100, 100)):
    img = np.zeros((200, 200), np.uint8)
    if kind == "circle":
        cv2.circle(img, center, size, 255, -1)
    elif kind in ("square", "triangle", "hexagon"):
        n = {"square": 4, "triangle": 3, "hexagon": 6}[kind]
        a = np.deg2rad(angle) + np.arange(n) * 2 * np.pi / n
        pts = np.stack([center[0] + size * np.cos(a), center[1] + size * np.sin(a)], 1).astype(np.int32)
        cv2.fillPoly(img, [pts], 255)
    elif kind == "bolt":
        box = cv2.boxPoints(((center[0], center[1]), (size * 2.6, size * 0.6), angle)).astype(np.int32)
        cv2.fillPoly(img, [box], 255)
    return img


def describe(c):
    A, P = cv2.contourArea(c), cv2.arcLength(c, True)
    (_, _), (w, h), ang = cv2.minAreaRect(c)
    hull = cv2.contourArea(cv2.convexHull(c))
    x, y, bw, bh = cv2.boundingRect(c)
    return {"area": A, "circularity": 4 * np.pi * A / P ** 2, "aspect": max(w, h) / max(min(w, h), 1),
            "extent": A / (bw * bh), "solidity": A / hull, "vertices": len(cv2.approxPolyDP(c, 0.03 * P, True))}


kinds = ["circle", "hexagon", "square", "triangle", "bolt"]
fig, axes = plt.subplots(1, 5, figsize=(22, 4.4))
table = []
for ax, k in zip(axes, kinds):
    im = make_shape(k, 55, 20)
    c = max(cv2.findContours(im, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    dsc = describe(c)
    table.append(dsc)
    ax.imshow(im, cmap="gray"); ax.axis("off")
    ax.set_title(f"{k}\ncirc {dsc['circularity']:.2f} | aspect {dsc['aspect']:.1f} | vertices {dsc['vertices']}")
plt.suptitle("Figure 5.1 - The same descriptors distinguish very different shapes", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
base = make_shape("hexagon", 55, 0)
variants = {"original": base, "rotated 37 deg": make_shape("hexagon", 55, 37), "scaled 0.5": make_shape("hexagon", 28, 0),
            "shifted": make_shape("hexagon", 55, 0, (70, 120)), "a square": make_shape("square", 55, 0)}
hu = {}
for name, im in variants.items():
    h = cv2.HuMoments(cv2.moments(im, binaryImage=True)).ravel()
    hu[name] = -np.sign(h) * np.log10(np.abs(h) + 1e-30)          # log scale makes the 7 values comparable
fig, axes = plt.subplots(1, 2, figsize=(17, 4.2), gridspec_kw={"width_ratios": [2.4, 1]})
x = np.arange(7)
for k, (name, v) in enumerate(hu.items()):
    axes[0].bar(x + (k - 2) * 0.16, v, 0.16, label=name)
axes[0].set_xticks(x, [f"phi{i + 1}" for i in range(7)]); axes[0].legend(); axes[0].set_ylabel("-sign * log10|phi|")
axes[0].set_title("Hu moments: identical for rotated/scaled/shifted hexagons, different for a square")
axes[1].axis("off")
scores = [[name, f"{cv2.matchShapes(base, im, cv2.CONTOURS_MATCH_I1, 0):.4f}"] for name, im in variants.items()]
axes[1].table(cellText=scores, colLabels=["compared with original", "matchShapes (0 = same)"], loc="center").scale(1, 1.6)
plt.suptitle("Figure 5.2 - Moment invariants for shape matching", fontweight="bold")
plt.tight_layout(); plt.show()

### 5.1 Finding notches with convexity defects
**Solidity** drops when a part has a missing chunk. `convexityDefects` goes further: it finds *where* the contour departs from its convex hull and
*how deep* each dent is. A healthy washer has no deep defects; a chipped one has one.

In [ ]:
def washer(chipped=False, dent_at=45):
    im = np.zeros((220, 220), np.uint8)
    cv2.circle(im, (110, 110), 90, 255, -1)
    if chipped:
        a = np.deg2rad(dent_at)
        cv2.circle(im, (int(110 + 95 * np.cos(a)), int(110 - 95 * np.sin(a))), 22, 0, -1)
    return im


fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, im) in zip(axes, [("good", washer()), ("chipped at 45 deg", washer(True)), ("chipped at 200 deg", washer(True, 200))]):
    c = max(cv2.findContours(im, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    hull_idx = cv2.convexHull(c, returnPoints=False)
    defects = cv2.convexityDefects(c, hull_idx)
    if defects is not None:
        defects = defects.reshape(-1, 4)        # (N, 1, 4) in OpenCV 4, (N, 4) in OpenCV 5
    vis = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR) // 2
    cv2.drawContours(vis, [cv2.convexHull(c)], -1, (0, 200, 0), 1)
    depths = [] if defects is None else defects[:, 3] / 256.0
    worst = 0.0
    if defects is not None:
        s, e, f, d = defects[np.argmax(depths)]
        worst = d / 256.0
        cv2.circle(vis, tuple(map(int, c[f][0])), 6, (0, 0, 255), -1)
        cv2.line(vis, tuple(map(int, c[s][0])), tuple(map(int, c[e][0])), (255, 200, 0), 2)
    verdict = "REJECT" if worst > 3 else "OK"
    ax.imshow(vis[..., ::-1]); ax.axis("off")
    ax.set_title(f"{name}: solidity {describe(c)['solidity']:.3f}, deepest defect {worst:.1f} px -> {verdict}",
                 color="#d93025" if verdict == "REJECT" else "#188038")
plt.suptitle("Figure 5.3 - Convexity defects locate notches (green = convex hull, red = deepest point)", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Case study: a complete coin inspection

Combining everything from Weeks 3–4: **white top-hat** to remove the uneven background → **Otsu** → **hole filling** (coins with dark centres) → remove border-touching blobs →
**distance-transform watershed** for touching coins → measure **equivalent diameters** → classify by size. This is the structure of many real
inspection programs: pre-process, segment, split, measure, decide.

In [ ]:
def fill_holes(binary):
    """Fill every hole by redrawing the outer contours filled."""
    out = np.zeros_like(binary)
    cs, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, cs, -1, 255, -1)
    return out


# 1. remove the uneven background with a white top-hat (median first: an opening is a minimum filter, sensitive to noise)
background = cv2.morphologyEx(cv2.medianBlur(coins, 5), cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (81, 81)))
tophat = cv2.subtract(coins, background)
# 2. threshold, clean, fill the dark coin centres
_, b = cv2.threshold(cv2.GaussianBlur(tophat, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
b = cv2.morphologyEx(b, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
b = fill_holes(b)            # (a closing here would smooth the outlines but merge touching coins: try it in E4)
n_before = cv2.connectedComponents(b)[0] - 1
# 3. drop tiny blobs and blobs touching the border (incomplete coins cannot be measured)
n_b, lab_b, st_b, _ = cv2.connectedComponentsWithStats(b)
for i in range(1, n_b):
    x, y, w, h, area = st_b[i]
    if area < 200 or x == 0 or y == 0 or x + w == b.shape[1] or y + h == b.shape[0]:
        b[lab_b == i] = 0
dist_c = cv2.distanceTransform(b, cv2.DIST_L2, 5)
# 4. markers: the top of each blob's distance hill, relative to THAT blob's own maximum (coins differ in size)
n_c, lab_c = cv2.connectedComponents(b)
pk = np.zeros_like(b)
for i in range(1, n_c):
    m = lab_c == i
    pk[m & (dist_c > 0.5 * dist_c[m].max())] = 255
nm, mk = cv2.connectedComponents(pk)
mk = mk + 1; mk[(b > 0) & (pk == 0)] = 0
ws_c = cv2.watershed(cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR), mk)
regions = [lbl for lbl in range(2, ws_c.max() + 1) if (ws_c == lbl).sum() >= 200]
all_d = [np.sqrt(4 * (ws_c == lbl).sum() / np.pi) for lbl in regions]
small_lim, large_lim = np.percentile(all_d, [33, 67])          # size classes from the data (tertiles)
diam = []
vis = cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR)
for lbl in regions:
    m = (ws_c == lbl).astype(np.uint8)
    area = m.sum()
    if area < 200:
        continue
    d_eq = np.sqrt(4 * area / np.pi)
    diam.append(d_eq)
    cy, cx = np.argwhere(m).mean(0).astype(int)
    size = "L" if d_eq > large_lim else ("M" if d_eq > small_lim else "S")
    cv2.drawContours(vis, cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0], -1, (0, 200, 0), 2)
    cv2.putText(vis, size, (cx - 6, cy + 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
fig, axes = plt.subplots(1, 3, figsize=(21, 5), gridspec_kw={"width_ratios": [1, 1.4, 1]})
axes[0].imshow(b, cmap="gray"); axes[0].set_title("top-hat, Otsu, opening, hole filling, border blobs removed"); axes[0].axis("off")
axes[1].imshow(vis[..., ::-1]); axes[1].set_title(f"{n_before} raw blobs -> filtering + watershed -> {len(diam)} coins, labelled S / M / L"); axes[1].axis("off")
axes[2].hist(diam, bins=15, color="#1a73e8"); axes[2].axvline(small_lim, color="k", ls="--"); axes[2].axvline(large_lim, color="k", ls="--")
axes[2].set_xlabel("equivalent diameter (px)"); axes[2].set_title("size distribution and S/M/L limits")
plt.suptitle("Figure 6.1 - From image to measured objects", fontweight="bold")
plt.tight_layout(); plt.show()

**Look critically at Figure 6.1.** All 24 coins are found, but several low-contrast coins (top row) have ragged outlines, so their measured
diameters are too small. The size classes are only as good as the segmentation. Two fixes: improve the image (a darker background or
dark-field lighting, Week 2A), or measure the outline with a model, e.g. fit a circle (`cv2.minEnclosingCircle`, Hough circles in Week 4A)
instead of counting pixels.

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Circularity
Return $4\pi A/P^2$ for an OpenCV contour (use `contourArea` and `arcLength`).

In [ ]:
def circularity(contour):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    circle = max(cv2.findContours(make_shape("circle", 60, 0), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    square = np.array([[[0, 0]], [[100, 0]], [[100, 100]], [[0, 100]]], np.int32)
    assert 0.85 < circularity(circle) <= 1.05, f"a digital circle should score close to 1, got {circularity(circle):.3f}"
    assert abs(circularity(square) - np.pi / 4) < 1e-6, f"a square should score pi/4 = 0.785, got {circularity(square):.3f}"


check("E1 circularity", _test_e1)

### E2. Count holes
Return a list with the number of holes of **every object** in a binary image (in any order). An object is any white region, including one lying inside another part's hole (like the pin in Figure 4.1). Use the contour hierarchy (`RETR_CCOMP` or `RETR_TREE`).

In [ ]:
def count_holes(binary):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    got = count_holes(parts)
    assert sorted(got) == [0, 0, 1, 2], f"the parts image has 4 objects with 2, 1, 0 and 0 (the pin) holes, you returned {got}"
    img = np.zeros((100, 300), np.uint8)
    for k, h in enumerate([3, 0, 5]):
        cv2.rectangle(img, (10 + 100 * k, 10), (90 + 100 * k, 90), 255, -1)
        for j in range(h):
            cv2.circle(img, (22 + 100 * k + 14 * j, 50), 4, 0, -1)
    assert sorted(count_holes(img)) == [0, 3, 5], f"expected holes [0, 3, 5] in some order, got {count_holes(img)}"


check("E2 count_holes", _test_e2)

### E3. Shape classifier
Return `"circle"`, `"triangle"`, `"square"`, `"hexagon"` or `"bolt"` for a contour, using descriptors from Section 5. It will be tested on 50 random shapes of different sizes and rotations.

In [ ]:
def classify_shape(contour):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    r = np.random.default_rng(5)
    correct = 0
    for _ in range(50):
        k = kinds[r.integers(len(kinds))]
        im = make_shape(k, int(r.integers(30, 70)), float(r.uniform(0, 180)))
        c = max(cv2.findContours(im, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
        correct += classify_shape(c) == k
    assert correct >= 45, f"{correct}/50 correct: aim for at least 45. Combine circularity, aspect ratio and vertex count."


check("E3 classify_shape", _test_e3)

### E4. Real objects (no self-check)
Segment `fruits.jpg` into individual fruits using any combination of this week's tools (k-means, flood fill, watershed). Report which fruit is the
largest by area and which is the most circular. Which steps would fail on a cluttered conveyor?

In [ ]:
# E4: your code here

## 8. Challenge: nuts, bolts and washers

Put a handful of small hardware (nuts, bolts, washers, screws) on a white sheet of paper and photograph it from above with even lighting.
1. Segment the parts (threshold + morphology), split any that touch (watershed).
2. Classify each part using circularity, aspect ratio and the number of holes.
3. Output a bill of materials: `washer: 3, nut: 2, bolt: 4`, and an annotated image.
4. Calibrate the camera (Week 2B) to report bolt lengths in millimetres.

In [ ]:
# Your challenge code here

## 9. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Two blobs touch only at a single diagonal pixel. How many components with 4-connectivity and with 8-connectivity?',
        "options": [
            '1 and 1',
            '2 and 1',
            '1 and 2',
            '2 and 2',
        ],
        "answer": '401c0cf6e5',
        "explain": 'Diagonal neighbours only connect under 8-connectivity.',
    },
    {
        "q": 'What does the distance transform compute for a foreground pixel?',
        "options": [
            'Its distance to the image centre',
            'Its distance to the nearest background pixel',
            'Its grey level',
            'The number of neighbours',
        ],
        "answer": '30b9d7680d',
        "explain": 'Object centres become maxima, which makes good watershed markers.',
    },
    {
        "q": 'Why does watershed without markers usually over-segment?',
        "options": [
            'It is too slow',
            'Every local minimum (including noise) starts its own region',
            'It ignores edges',
            'It needs colour images',
        ],
        "answer": 'eb70797f66',
        "explain": 'Marker-controlled watershed fixes this.',
    },
    {
        "q": 'With RETR_TREE, what is a child of an outer contour?',
        "options": [
            'A neighbouring object',
            'A hole inside it',
            'Its convex hull',
            'A smaller copy',
        ],
        "answer": 'd4496cf54f',
        "explain": 'An object inside that hole would be a grandchild.',
    },
    {
        "q": 'Which descriptor equals 1 for a perfect circle and is scale-invariant?',
        "options": [
            'Area',
            'Perimeter',
            'Circularity 4 pi A / P^2',
            'Extent',
        ],
        "answer": '4fac90349b',
        "explain": 'A square scores pi/4 = 0.785.',
    },
    {
        "q": 'A washer has a small chip on its rim. Which descriptor changes most clearly?',
        "options": [
            'Area',
            'Solidity / convexity defects',
            'Aspect ratio',
            'Centroid',
        ],
        "answer": '887e52a738',
        "explain": 'The contour dips inside its convex hull at the chip.',
    },
    {
        "q": 'Why are Hu moments useful for shape matching?',
        "options": [
            'They are fast to compute only',
            'They are invariant to translation, rotation and scale',
            'They encode colour',
            'They require no segmentation',
        ],
        "answer": 'ea723138d2',
        "explain": 'cv2.matchShapes compares shapes using them.',
    },
], title='Week 04B quiz')

## 10. Exit ticket (reflection)
1. Describe a situation where k-means colour clustering gives a better segmentation than thresholding, and one where it is worse.
2. In the coin case study, which step would you change first if the coins were on a textured background?
3. Name two measurements a robot needs from a segmented part to pick it up, and the descriptor that provides each.

## 11. What's new (2025–26)

> - **Instance-segmentation networks** (Mask R-CNN, YOLO-seg, SAM; Week 11) now split touching objects directly, but the shape descriptors in this
>   notebook are still how their masks are **measured** and accepted or rejected on a production line.
> - **Superpixels** (e.g. SLIC) and graph-based region methods remain popular as fast pre-segmentation for annotation tools.
> - Classical blob analysis runs on **smart cameras** at thousands of parts per minute. Vendor tools (e.g. HALCON, VisionPro, the open-source
>   OpenCV-based pipelines) are built on exactly these operators.

## 12. References and further exploration

**Seminal papers**
- Hu, M.-K. (1962). *Visual pattern recognition by moment invariants.* IRE Trans. Information Theory 8(2). [doi:10.1109/TIT.1962.1057692](https://doi.org/10.1109/TIT.1962.1057692)
- Vincent, L. & Soille, P. (1991). *Watersheds in digital spaces: an efficient algorithm based on immersion simulations.* IEEE TPAMI 13(6). [doi:10.1109/34.87344](https://doi.org/10.1109/34.87344)
- Suzuki, S. & Abe, K. (1985). *Topological structural analysis of digitized binary images by border following* (the algorithm behind `findContours`). CVGIP 30(1). [doi:10.1016/0734-189X(85)90016-7](https://doi.org/10.1016/0734-189X(85)90016-7)
- Douglas, D. H. & Peucker, T. K. (1973). *Algorithms for the reduction of the number of points required to represent a digitized line.* Cartographica 10(2).
- Lloyd, S. (1982). *Least squares quantization in PCM* (k-means). IEEE Trans. Information Theory 28(2). [doi:10.1109/TIT.1982.1056489](https://doi.org/10.1109/TIT.1982.1056489)

**Textbooks**
- Gonzalez & Woods (2018), ch. 10 (region-based segmentation, watershed) and ch. 11 (feature extraction: boundary and region descriptors).
- Steger, Ulrich & Wiedemann (2018). *Machine Vision Algorithms and Applications*, 2nd ed., §3.5 (feature extraction) and §3.6 (morphology).
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §7.5 (segmentation). [szeliski.org/Book](https://szeliski.org/Book/)

**Documentation and interactive**
- OpenCV tutorials: *Contours* (features, properties, hierarchy), *Image segmentation with watershed*, *K-means clustering*. [docs.opencv.org](https://docs.opencv.org/4.x/d3/d05/tutorial_py_table_of_contents_contours.html)
- scikit-image: `measure.regionprops` (dozens of region properties in one call). [scikit-image.org](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops)

**Images:** `coins` from scikit-image (public domain); `fruits.jpg` from the OpenCV sample data. Parts, shapes and washers are generated in code.

## 13. Key takeaways
- **Connected components** turn a mask into objects; connectivity (4/8) matters for thin or diagonal structures.
- **Region growing** segments one seeded object; **k-means** clusters pixels by colour.
- **Distance transform + marker-controlled watershed** splits touching objects; markers prevent over-segmentation.
- **Contours** and their **hierarchy** describe boundaries and holes.
- **Shape descriptors** (circularity, aspect ratio, solidity, moments, Hu invariants, convexity defects) support classification, gauging and defect detection.

**Next week (05A):** from regions to **keypoints**: scale space, corner detectors, SIFT and ORB features.